### Purpose
This notebook loads the raw CSV dataset and saves it in Parquet format for faster and more consistent processing in the next steps.
https://www.kaggle.com/datasets/milanzdravkovic/pharma-sales-data
<br> --- --- --- <br>
(Este notebook carrega o dataset bruto em CSV e o salva em formato Parquet, para permitir processamento mais rápido e consistente nas próximas etapas.)
<br>
📝 Language: Technical documentation is maintained in English to ensure consistency and ease of maintenance. Bilingual support (Portuguese/English) is available exclusively within the notebooks.
<br> --- --- --- <br>
📝 Idioma: A documentação é mantida em inglês para garantir consistência técnica e evitar retrabalho na documentação. Isto aplica-se somente aos notebooks.

### ⚠️ Attention Atenção
📝 There may still be some notes and comments in Portuguese, which will be updated to English over time.
<br> --- --- --- <br>
📝 É possível que ainda existam algumas observações e notas em português, as quais serão gradualmente traduzidas para o inglês.


##### ⚙️The project follows this logic:
**raw data -> [feature engineering + anomaly detection] -> baseline model -> models -> Essemble Stack performance benchmarking**

##### ⚙️Execution order
📝01_data_loading.ipynb -> 📝02_feature_engineering.ipynb -> 📝05_BaseLine.ipynb -> 📝06_Models.ipynb 
-> 📝07_Interpretability_SHAP_Analysis.ipynb -> 📝08_performance_benchmarking.ipynb 


In [1]:
from __future__ import annotations

import argparse
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sys
from sklearn.ensemble import IsolationForest

PROJECT_ROOT = Path().resolve().parent
ROOT = Path().resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))


from src.config.config import DATA_RAW,DATA_FEATURES,DATA_EXPORTS,DATA_PROCESSED
from src.loaders.csv_loader import CSVLoader
from src.repository.parquet_repository import ParquetRepository
from src.dataProcessing.data_quality import DataQuality
from src.features.features import calendar_diagnostics


csvLoader = CSVLoader()
parket_data = ParquetRepository(DATA_FEATURES)


In [2]:
ATC_CATEGORIES = ("M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06")
IF_FEATURES = (
    "qty_lag_origin", "qty_lag_origin_7", "qty_roll_mean_7",
    "qty_roll_std_7", "qty_roll_mean_28", "qty_roll_std_28",
    "qty_mean_ratio_7_28", "qty_last_zscore_28", "qty_zero_ratio_28",
    "weekday", "annual_sin", "annual_cos",
)
TARGET_COL = "quantity"
NON_PREDICTOR_COLUMNS = {
    "date", "category", "forecast_origin", "horizon_days", TARGET_COL, "split",
}

In [3]:
def load_salesxx(csv_path: Path, category: str) -> pd.Series:
    """Compatibilidade com código anterior; no pipeline principal o CSV é lido 1 vez."""
    return pd.read_csv(csv_path, usecols=["datum", category]), category


def make_features(series: pd.Series, category: str, horizon: int) -> pd.DataFrame:
    """Gera features por horizonte, com corte estrito no último dia conhecido t-h."""

    if horizon <= 0:
        raise ValueError("horizon deve ser >= 1")
    if not isinstance(series.index, pd.DatetimeIndex) or not series.index.is_monotonic_increasing:
        raise ValueError("series deve usar índice DatetimeIndex crescente")
    if not series.index.is_unique:
        raise ValueError("A série contém datas repetidas")
    
    idx = series.index
    y = series.astype(float)
    df = pd.DataFrame(index=idx)

    df["date"] = idx
    df["category"] = category
    df["horizon_days"] = int(horizon)
    df["forecast_origin"] = idx - pd.Timedelta(days=horizon)
    df[TARGET_COL] = y  # Rótulo disponível somente no treinamento retrospectivo

    # Datas futuras do calendário são conhecidas na origem.
    df["year"] = idx.year
    df["month"] = idx.month
    df["quarter"] = idx.quarter
    df["weekday"] = idx.dayofweek
    df["dayofyear"] = idx.dayofyear
    df["weekofyear"] = idx.isocalendar().week.to_numpy(dtype=int)
    df["is_weekend"] = (idx.dayofweek >= 5).astype(int)


    df["weekday_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7) 
    df["weekday_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    df["annual_sin"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    df["annual_cos"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)

    known = y.shift(horizon)
    for back in (0, 1, 7, 14, 28, 56):
        name = "qty_lag_origin" if back == 0 else f"qty_lag_origin_{back}"
        df[name] = known.shift(back)

    for window in (7, 14, 28, 56):
        rolling = known.rolling(window=window, min_periods=window)
        df[f"qty_roll_mean_{window}"] = rolling.mean()
        df[f"qty_roll_std_{window}"] = rolling.std(ddof=0)
        if window in (7, 28):
            df[f"qty_roll_sum_{window}"] = rolling.sum()
            df[f"qty_roll_min_{window}"] = rolling.min()
            df[f"qty_roll_max_{window}"] = rolling.max()
    
    # avoids division by zero; evita divisao por zero
    eps = 1e-6

    mean7 = df["qty_roll_mean_7"]
    mean28 = df["qty_roll_mean_28"]
    std28 = df["qty_roll_std_28"]
    df["qty_mean_ratio_7_28"] = mean7 / (mean28 + eps)
    df["qty_mean_delta_7_28"] = mean7 - mean28
    df["qty_last_ratio_28"] = known / (mean28 + eps)
    df["qty_last_zscore_28"] = (known - mean28) / (std28 + eps)
    df["qty_cv_28"] = std28 / (mean28 + eps)
    df["qty_zero_ratio_28"] = known.eq(0).rolling(28, min_periods=28).mean()
    df["qty_mean_ratio_14_56"] = df["qty_roll_mean_14"] / (
        df["qty_roll_mean_56"] + eps
    )

    df = df.dropna(axis=0).reset_index(drop=True)
    if df.empty:
        raise ValueError("Histórico insuficiente para gerar o horizonte solicitado")
    return df


def feature_columns(df: pd.DataFrame) -> list[str]:
    """Preditoras LightGBM, incluindo anomaly_score e is_anomaly após o IF."""
    return [c for c in df.columns if c not in NON_PREDICTOR_COLUMNS]

In [4]:
def fit_if_train_only(
    df: pd.DataFrame, cutoff: pd.Timestamp, contamination: float = 0.03,
    random_state: int = 42,
) -> tuple[pd.DataFrame, dict]:
    """Ajusta IF APENAS no treino, pontua treino/teste sem usar a venda alvo."""
    if not (0 < contamination < 0.5):
        raise ValueError("contamination precisa estar entre 0 e 0.5")
    out = df.copy()
    train_mask = out["date"] <= cutoff
    if train_mask.sum() < 200 or (~train_mask).sum() < 1:
        raise ValueError("Quantidade insuficiente de dados no treino ou teste")
    for col in IF_FEATURES:
        if col not in out.columns:
            raise KeyError(f"Feature IF não encontrada: {col}")
    X_train = out.loc[train_mask, list(IF_FEATURES)]
    X_all = out.loc[:, list(IF_FEATURES)]
    if not np.isfinite(X_all.to_numpy(dtype=float)).all():
        raise ValueError("Há valores não finitos nas features IF")

    model = IsolationForest(
        n_estimators=250,
        max_samples="auto",
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X_train)
    raw_train = -model.score_samples(X_train)  # maior = mais anômalo
    raw_all = -model.score_samples(X_all)
    train_min = float(raw_train.min())
    train_max = float(raw_train.max())
    denom = max(train_max - train_min, 1e-12)
    score_train = np.clip((raw_train - train_min) / denom, 0.0, 1.0)
    score_all = np.clip((raw_all - train_min) / denom, 0.0, 1.0)
    threshold = float(np.quantile(score_train, 1.0 - contamination))

    out["anomaly_score"] = score_all
    out["is_anomaly"] = (out["anomaly_score"] >= threshold).astype(np.int8)
    out["split"] = np.where(train_mask, "train", "test")
    artifact = {
        "model": model,
        "feature_columns": list(IF_FEATURES),
        "score_min_train": train_min,
        "score_max_train": train_max,
        "threshold": threshold,
        "contamination": float(contamination),
        "train_cutoff": cutoff.strftime("%Y-%m-%d"),
        "semantics": "Anomaly of historical context at forecast_origin; NOT target-day anomaly",
    }
    return out, artifact


def build_features_dataframe(
    df_salesdaily: pd.DataFrame,
    category: str = "N02BE",
    horizons: tuple[int, ...] | list[int] = (7, 15),
    test_months: int = 6,
    contamination: float = 0.03,
) -> tuple[pd.DataFrame, dict[int, dict]]:
    """Constrói o DataFrame final de features e modelos de anomalia em memória.

    Recebe os dados brutos e retorna um DataFrame completo unificado e artefatos do IF. 
    Não realiza operações de leitura/escrita em disco; o chamador deve decidir quando 
    persistir os dados. A mesma categoria produz uma linha por `(date, horizon_days)`.
    """
    if test_months <= 0:
        raise ValueError("test_months deve ser positivo")
    if not horizons or any(not isinstance(h, int) or h <= 0 for h in horizons):
        raise ValueError("horizons deve conter inteiros positivos")

    series = DataQuality.sales_series_from_dataframe(df_salesdaily, category,"quantity")
    cutoff = series.index.max() - pd.DateOffset(months=test_months)
    frames: list[pd.DataFrame] = []
    artifacts: dict[int, dict] = {}

    for horizon in sorted(set(horizons)):
        features = make_features(series, category, horizon)
        result, artifact = fit_if_train_only(features, cutoff, contamination)
        ordered = (
            ["date", "category", "horizon_days", "forecast_origin", "split", TARGET_COL]
            + feature_columns(result)
        )
        frames.append(result.loc[:, ordered])
        artifacts[horizon] = artifact

    df_features = pd.concat(frames, ignore_index=True)
    df_features = df_features.sort_values(["horizon_days", "date"]).reset_index(drop=True)
    return df_features, artifacts

In [5]:
df_salesdaily = csvLoader.create_dataframe(DATA_RAW,"salesdaily.csv")
#df_salesdaily = parket_data.load("anomaly_predictions.parquet")

df_features, artifacts = build_features_dataframe(
    df_salesdaily, category="N02BE", horizons=(7,15),
    test_months=6, contamination=0.03,
)

C:\Users\AlexK\AppData\Local\Temp\ipykernel_26500\969800549.py:23: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  df["forecast_origin"] = idx - pd.Timedelta(days=horizon)
C:\Users\AlexK\AppData\Local\Temp\ipykernel_26500\969800549.py:23: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  df["forecast_origin"] = idx - pd.Timedelta(days=horizon)


In [6]:
def create_anomaly_features(
    df,
    group_col="split",
    date_col="date"
):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    df = df.sort_values([group_col, date_col])

    for janela in [7, 15]:

        df[f"anomaly_score_mean{janela}"] = (
            df.groupby(group_col)["anomaly_score"]
            .transform(
                lambda s: s.rolling(
                    janela,
                    min_periods=janela
                ).mean()
            )
        )

        df[f"is_anomaly_rate{janela}"] = (
            df.groupby(group_col)["is_anomaly"]
            .transform(
                lambda s: s.rolling(
                    janela,
                    min_periods=janela
                ).mean()
            )
        )

    return df

In [7]:
df_features_anomaly = create_anomaly_features(df_features,"split","date").copy()


In [8]:
features_required = [
    "anomaly_score_mean7",
    "anomaly_score_mean15"
]

# Identifica registros sem histórico suficiente
mask_missing = df_features_anomaly[features_required].isna().any(axis=1)

# Permite o valor "none", inclusive se split for categórico
df_features_anomaly["split"] = df_features_anomaly["split"].astype("string")

# Marca registros inválidos para treinamento/avaliação
df_features_anomaly.loc[mask_missing, "split"] = "none"

# Resumo
print(df_features_anomaly["split"].value_counts(dropna=False))

split
train    3698
test      352
none       28
Name: count, dtype: Int64


In [70]:
df_features_anomaly.head(100)

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly,anomaly_score_mean7,is_anomaly_rate7,anomaly_score_mean15,is_anomaly_rate15
1860,2019-04-09,N02BE,7,2019-04-02,none,12.0,2019,4,2,1,...,-0.575679,0.364572,0.000000,0.856451,0.107515,0,NaN,NaN,NaN,NaN
3895,2019-04-09,N02BE,15,2019-03-25,none,12.0,2019,4,2,1,...,-0.348076,0.375162,0.000000,0.920462,0.025314,0,NaN,NaN,NaN,NaN
1861,2019-04-10,N02BE,7,2019-04-03,none,34.0,2019,4,2,2,...,-1.028732,0.377550,0.000000,0.837558,0.116822,0,NaN,NaN,NaN,NaN
3896,2019-04-10,N02BE,15,2019-03-26,none,34.0,2019,4,2,2,...,0.396549,0.371568,0.000000,0.930312,0.028274,0,NaN,NaN,NaN,NaN
1862,2019-04-11,N02BE,7,2019-04-04,none,15.9,2019,4,2,3,...,0.748972,0.346883,0.000000,0.885301,0.107962,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3942,2019-05-26,N02BE,15,2019-05-11,test,30.0,2019,5,2,6,...,-0.393220,0.515861,0.071429,0.830408,0.298657,0,0.252342,0.0,0.267832,0.0
1908,2019-05-27,N02BE,7,2019-05-20,test,19.0,2019,5,2,0,...,-0.086425,0.409831,0.035714,0.998092,0.179414,0,0.229459,0.0,0.256486,0.0
3943,2019-05-27,N02BE,15,2019-05-12,test,19.0,2019,5,2,0,...,1.511120,0.524799,0.071429,0.962762,0.340601,0,0.255551,0.0,0.258046,0.0
1909,2019-05-28,N02BE,7,2019-05-21,test,18.2,2019,5,2,1,...,-0.491414,0.410944,0.035714,0.961147,0.148325,0,0.239441,0.0,0.242071,0.0


In [9]:
# 1. Garante que os dados estejam ordenados cronologicamente
df_features_anomaly = df_features_anomaly.sort_values('date').reset_index(drop=True)

# 2. Transforma zeros e valores negativos em NaN
df_features_anomaly['quantity'] = df_features_anomaly['quantity'].mask(df_features_anomaly['quantity'] <= 0, np.nan)

# 3. Interpola os NaNs usando a média do vizinho anterior e posterior (método linear)
df_features_anomaly['quantity'] = df_features_anomaly['quantity'].interpolate(method='linear')

In [10]:
# Histórico diário original
daily = df_features_anomaly.rename(
    columns={"datum": "date"}
).copy()

daily["date"] = pd.to_datetime(
    daily["date"]
).dt.normalize()

# Transformar categorias ATC em linhas
daily = daily.melt(
    id_vars="date",
    var_name="category",
    value_name="quantity_real"
)

# Dataset supervisionado atual
df = df_features_anomaly.copy()

df["date"] = pd.to_datetime(
    df["date"]
).dt.normalize()

# Guardar o alvo anterior para auditoria
df["quantity_original"] = df["quantity"]

# Calcular a data futura de cada horizonte
df["target_date"] = (
    df["date"] +
    pd.to_timedelta(df["horizon_days"], unit="D")
)

# 1. Histórico original: uma linha por data e categoria ATC
daily = df_salesdaily[
    ["datum", *ATC_CATEGORIES]
].rename(columns={"datum": "date"}).copy()

daily["date"] = pd.to_datetime(
    daily["date"], format="%m/%d/%Y"
).dt.normalize()

daily = daily.melt(
    id_vars="date",
    value_vars=list(ATC_CATEGORIES),
    var_name="category",
    value_name="quantity_real"
)

# 2. Verificar unicidade das chaves
assert not daily.duplicated(
    ["date", "category"]
).any(), "Existem duplicações no histórico diário"

# 3. Dataset supervisionado
df = df_features_anomaly.copy()
df["date"] = pd.to_datetime(df["date"]).dt.normalize()

df["quantity_original"] = df["quantity"]

# IMPORTANTE:
# make_features() já usa date como a data do alvo.
df["target_date"] = df["date"]

# 4. Merge com os valores reais
df = df.merge(
    daily.rename(columns={"date": "target_date"}),
    on=["target_date", "category"],
    how="left",
    validate="many_to_one"
)

# 5. Validar correspondência
assert df["quantity_real"].notna().all(), (
    "Existem datas/categorias sem venda real"
)

# 6. Utilizar o valor observado como alvo
df["quantity"] = df["quantity_real"]

# 7. Manter somente treino e teste
df = df[df["split"].isin(["train", "test"])].copy()

print("Total de registros:", len(df))
print("Alvos ausentes:", df["quantity"].isna().sum())
#---

# A coluna quantity passa a representar a venda futura
df["quantity"] = df["quantity_real"]

# Descartar linhas sem alvo futuro observado
df = df.dropna(subset=["quantity"]).copy()

# Usar apenas treino/teste
df = df[df["split"].isin(["train", "test"])].copy()

Total de registros: 4050
Alvos ausentes: 0


In [11]:
# Full artifact retained for API/monitoring use.
parket_data.set_base_path(DATA_EXPORTS)
parket_data.save(df_features_anomaly, "anomaly_predictions_api.parquet", compression="zstd")

parket_data.set_base_path(DATA_FEATURES)
parket_data.save(df_features_anomaly, "anomaly_predictions.parquet", compression="zstd")

[OK] Arquivo salvo em: D:\PROJETOS\git_repo\PharmaDemand-ml\data\exports\anomaly_predictions_api.parquet
[OK] Arquivo salvo em: D:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet


In [12]:
df_load = parket_data.load("anomaly_predictions.parquet")

print (df_load.dtypes)

colunas = {"anomaly_score_mean7", "anomaly_score_mean15"}
if colunas.issubset(df_load.columns):
    print("As duas colunas existem!")
else:
    faltantes = colunas - set(df_load.columns)
    print(f"Colunas ausentes: {faltantes}")

[OK] Arquivo carregado: D:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
date                    datetime64[ns]
category                        object
horizon_days                     int64
forecast_origin         datetime64[ns]
split                   string[python]
quantity                       float64
year                             int32
month                            int32
quarter                          int32
weekday                          int32
dayofyear                        int32
weekofyear                       int64
is_weekend                       int64
weekday_sin                    float64
weekday_cos                    float64
annual_sin                     float64
annual_cos                     float64
qty_lag_origin                 float64
qty_lag_origin_1               float64
qty_lag_origin_7               float64
qty_lag_origin_14              float64
qty_lag_origin_28              float64
qty_lag_origin_56              float64


In [87]:
df_load.head(10)

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly,anomaly_score_mean7,is_anomaly_rate7,anomaly_score_mean15,is_anomaly_rate15
0,2014-03-06,N02BE,7,2014-02-27,none,44.1,2014,3,1,3,...,-0.174629,0.359129,0.0,1.047649,0.087633,0,NaN,NaN,NaN,NaN
1,2014-03-07,N02BE,7,2014-02-28,none,25.0,2014,3,1,4,...,0.096526,0.352603,0.0,1.043780,0.080584,0,NaN,NaN,NaN,NaN
2,2014-03-08,N02BE,7,2014-03-01,none,27.6,2014,3,1,5,...,1.139263,0.356460,0.0,1.080984,0.112373,0,NaN,NaN,NaN,NaN
3,2014-03-09,N02BE,7,2014-03-02,none,11.3,2014,3,1,6,...,-2.189715,0.372513,0.0,1.039697,0.274512,0,NaN,NaN,NaN,NaN
4,2014-03-10,N02BE,7,2014-03-03,none,47.0,2014,3,1,0,...,-0.983335,0.382841,0.0,1.028270,0.095044,0,NaN,NaN,NaN,NaN
5,2014-03-11,N02BE,7,2014-03-04,none,30.4,2014,3,1,1,...,0.819677,0.380388,0.0,1.006306,0.106161,0,NaN,NaN,NaN,NaN
6,2014-03-12,N02BE,7,2014-03-05,none,19.6,2014,3,1,2,...,-0.778093,0.387062,0.0,0.914075,0.042547,0,0.114122,0.0,NaN,NaN
7,2014-03-13,N02BE,7,2014-03-06,none,15.5,2014,3,1,3,...,0.592262,0.387364,0.0,0.954226,0.062473,0,0.110528,0.0,NaN,NaN
8,2014-03-14,N02BE,7,2014-03-07,none,16.0,2014,3,1,4,...,-0.798389,0.382694,0.0,0.968432,0.047920,0,0.105861,0.0,NaN,NaN
9,2014-03-14,N02BE,15,2014-02-27,none,16.0,2014,3,1,4,...,-0.174629,0.359129,0.0,1.047649,0.081521,0,0.101454,0.0,NaN,NaN


In [13]:
df_load[(df_load['quantity'] <= 0)]

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly,anomaly_score_mean7,is_anomaly_rate7,anomaly_score_mean15,is_anomaly_rate15


In [14]:
# Carregar dados originais
dados = df_load.copy()

dados["date"] = pd.to_datetime(dados["date"]).dt.normalize()

dados = dados[
    dados["split"].isin(["train", "test"]) &
    dados["horizon_days"].isin([7, 15])
].copy()

# Data que cada horizonte está tentando prever
dados["target_date"] = (
    dados["date"]
    + pd.to_timedelta(dados["horizon_days"], unit="D")
)

# D+7
d7 = dados[dados["horizon_days"] == 7][
    ["date", "target_date", "category", "split", "quantity"]
].rename(columns={
    "date": "origem_D7",
    "split": "split_D7",
    "quantity": "quantity_D7"
})

# D+15
d15 = dados[dados["horizon_days"] == 15][
    ["date", "target_date", "category", "split", "quantity"]
].rename(columns={
    "date": "origem_D15",
    "split": "split_D15",
    "quantity": "quantity_D15"
})

# Cruzar pela MESMA categoria e data-alvo
comparacao = d7.merge(
    d15,
    on=["category", "target_date"],
    how="inner",
    validate="one_to_one"
)

# Calcular diferenças
comparacao["diferenca"] = (
    comparacao["quantity_D7"] -
    comparacao["quantity_D15"]
)

comparacao["diferenca_absoluta"] = (
    comparacao["diferenca"].abs()
)

comparacao["divergente"] = ~np.isclose(
    comparacao["quantity_D7"],
    comparacao["quantity_D15"],
    atol=1e-6,
    rtol=0
)

# Somente registros divergentes
erros = comparacao[
    comparacao["divergente"]
].sort_values("target_date")

print("Total de comparações:", len(comparacao))
print("Total de divergências:", len(erros))

display(erros.head(20))

Total de comparações: 2010
Total de divergências: 1996


,origem_D7,target_date,category,split_D7,quantity_D7,origem_D15,split_D15,quantity_D15,diferenca,diferenca_absoluta,divergente
0,2014-03-25,2014-04-01,N02BE,train,38.6,2014-03-17,train,37.0,1.6,1.6,True
1,2014-03-26,2014-04-02,N02BE,train,6.7,2014-03-18,train,20.1,-13.4,13.4,True
2,2014-03-27,2014-04-03,N02BE,train,20.7,2014-03-19,train,21.8,-1.1,1.1,True
3,2014-03-28,2014-04-04,N02BE,train,11.0,2014-03-20,train,38.2,-27.2,27.2,True
4,2014-03-29,2014-04-05,N02BE,train,24.0,2014-03-21,train,40.0,-16.0,16.0,True
5,2014-03-30,2014-04-06,N02BE,train,12.0,2014-03-22,train,26.3,-14.3,14.3,True
6,2014-03-31,2014-04-07,N02BE,train,15.4,2014-03-23,train,19.4,-4.0,4.0,True
7,2014-04-01,2014-04-08,N02BE,train,27.5,2014-03-24,train,23.6,3.9,3.9,True
8,2014-04-02,2014-04-09,N02BE,train,23.0,2014-03-25,train,38.6,-15.6,15.6,True
9,2014-04-03,2014-04-10,N02BE,train,19.2,2014-03-26,train,6.7,12.5,12.5,True


In [15]:
dados[(dados['date'] == '2014-04-01')]

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly,anomaly_score_mean7,is_anomaly_rate7,anomaly_score_mean15,is_anomaly_rate15,target_date
44,2014-04-01,N02BE,7,2014-03-25,train,27.5,2014,4,2,1,...,0.405525,0.0,0.801923,0.097637,0,0.117682,0.0,0.106096,0.0,2014-04-08
45,2014-04-01,N02BE,15,2014-03-17,train,27.5,2014,4,2,1,...,0.449647,0.0,0.881141,0.202236,0,0.142515,0.0,0.107651,0.0,2014-04-16


In [16]:
dados["date"] = pd.to_datetime(dados["date"]).dt.normalize()

alvo = pd.Timestamp("2014-04-01")

filtro = (
    (dados["category"] == "N02BE") &
    (
        ((dados["date"] == alvo - pd.Timedelta(days=7)) &
         (dados["horizon_days"] == 7))
        |
        ((dados["date"] == alvo - pd.Timedelta(days=15)) &
         (dados["horizon_days"] == 15))
    )
)

resultado = dados.loc[
    filtro,
    ["date", "category", "horizon_days", "quantity", "split"]
].copy()

resultado["target_date"] = (
    resultado["date"] +
    pd.to_timedelta(resultado["horizon_days"], unit="D")
)

display(resultado.sort_values("horizon_days"))

C:\Users\AlexK\AppData\Local\Temp\ipykernel_26500\1089204103.py:8: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  ((dados["date"] == alvo - pd.Timedelta(days=7)) &
C:\Users\AlexK\AppData\Local\Temp\ipykernel_26500\1089204103.py:11: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  ((dados["date"] == alvo - pd.Timedelta(days=15)) &


,date,category,horizon_days,quantity,split,target_date
30,2014-03-25,N02BE,7,38.6,train,2014-04-01
14,2014-03-17,N02BE,15,37.0,train,2014-04-01


In [17]:
display(
    comparacao[
        (comparacao["category"] == "N02BE") &
        (comparacao["target_date"] == "2014-04-01")
    ]
)

,origem_D7,target_date,category,split_D7,quantity_D7,origem_D15,split_D15,quantity_D15,diferenca,diferenca_absoluta,divergente
0,2014-03-25,2014-04-01,N02BE,train,38.6,2014-03-17,train,37.0,1.6,1.6,True


In [19]:
def _validate_frame(df: pd.DataFrame, target_col: str, check_origin: bool = True) -> pd.DataFrame:
    required = {"date", target_col}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes: {sorted(missing)}")
    f = df.copy()
    f["date"] = pd.to_datetime(f["date"], errors="raise")
    if f["date"].dt.tz is not None:
        raise ValueError("Datas com timezone não são suportadas; padronize antes.")
    if (f["date"] != f["date"].dt.normalize()).any():
        raise ValueError("'date' deve conter somente dias (00:00:00).")
    f[target_col] = pd.to_numeric(f[target_col], errors="raise")
    v = f[target_col].to_numpy(dtype=float)
    if not np.isfinite(v).all() or (v < 0).any():
        raise ValueError("O alvo contém NaN, infinito ou valores negativos.")
    if f["date"].duplicated().any():
        raise ValueError("Datas duplicadas. Filtre category e horizon_days antes de treinar.")
    f = f.sort_values("date").reset_index(drop=True)
    if f.empty:
        raise ValueError("DataFrame vazio após filtragem.")
    expected = pd.date_range(f["date"].iloc[0], f["date"].iloc[-1], freq="D")
    if not pd.DatetimeIndex(f["date"]).equals(expected):
        missing_dates = expected.difference(pd.DatetimeIndex(f["date"]))
        raise ValueError(f"Série irregular, {len(missing_dates)} dias ausentes. Não preencha com zero automaticamente.")
    if check_origin and "forecast_origin" in f.columns and "horizon_days" in f.columns:
        f["forecast_origin"] = pd.to_datetime(f["forecast_origin"], errors="raise")
        ideal = f["date"] - pd.to_timedelta(f["horizon_days"], unit="D")
        if f["forecast_origin"].isna().any() or not f["forecast_origin"].equals(ideal.rename("forecast_origin")):
            raise ValueError("forecast_origin inconsistente com date - horizon_days.")
    return f

In [26]:
# Validando se existe datas duplicadas para a mesma categoria e horizon_days
# vamos pegar a categoria alvo do projeto N02BE e 7 dias
df_features_check = df_features.copy()

df_features_check = df_features_check[(df_features_check["category"] == "N02BE") & (df_features_check["horizon_days"] == 7)]

train = _validate_frame(df_features_check.loc[df_features_check["split"] == "train"], "quantity")
test = _validate_frame(df_features_check.loc[df_features_check["split"] == "test"], "quantity")

if len(train) < 365 or len(test) < 15:
    raise ValueError("Histórico insuficiente (requer >=365 dias de treino e >=h dias de teste).")

In [25]:
# Full artifact retained for API/monitoring use.
parket_data.set_base_path(DATA_EXPORTS)
parket_data.save(df_features, "anomaly_predictions_api.parquet", compression="zstd")

parket_data.set_base_path(DATA_FEATURES)
parket_data.save(df_features, "anomaly_predictions.parquet", compression="zstd")

[OK] Arquivo salvo em: D:\PROJETOS\git_repo\PharmaDemand-ml\data\exports\anomaly_predictions_api.parquet
[OK] Arquivo salvo em: D:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet
